In [169]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [170]:
from google.colab import drive

drive.mount('/content/drive')
!cp -r -n /content/drive/MyDrive/hustleflow_data/ /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [171]:
train_csv = '/content/hustleflow_data/train_raw.csv'
test_csv = '/content/hustleflow_data/test_raw.csv'

In [172]:
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

In [173]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   EmpNumber                     960 non-null    object
 1   Age                           960 non-null    int64 
 2   Gender                        960 non-null    object
 3   EducationBackground           960 non-null    object
 4   MaritalStatus                 960 non-null    object
 5   EmpDepartment                 960 non-null    object
 6   EmpJobRole                    960 non-null    object
 7   BusinessTravelFrequency       960 non-null    object
 8   DistanceFromHome              960 non-null    int64 
 9   EmpEducationLevel             960 non-null    int64 
 10  EmpEnvironmentSatisfaction    960 non-null    int64 
 11  EmpHourlyRate                 960 non-null    int64 
 12  EmpJobInvolvement             960 non-null    int64 
 13  EmpJobLevel         

Preprocess

In [174]:
X_train = train_df.drop(columns=['PerformanceRating', 'EmpNumber'])
y_train = train_df['PerformanceRating']
X_test = test_df.drop(columns=['PerformanceRating', 'EmpNumber'])
y_test = test_df['PerformanceRating']

In [176]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

cat_features = X_train.select_dtypes(include=['object']).columns
num_features = X_train.select_dtypes(include=np.number).columns

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features)
])

Model pipeline

In [177]:
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline

model = Pipeline([
    ('preprocess', preprocessor),
    ('svc', SVC(probability=True, random_state=42, class_weight='balanced'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

In [188]:
from sklearn.metrics import classification_report, roc_auc_score

print('ROC AUC score: ', roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted'))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

ROC AUC score:  0.9080514692294578

Classification report:
              precision    recall  f1-score   support

           0       0.49      0.82      0.62        39
           1       0.93      0.76      0.84       175
           2       0.56      0.69      0.62        26

    accuracy                           0.76       240
   macro avg       0.66      0.76      0.69       240
weighted avg       0.82      0.76      0.78       240



Hyperparameter Tuning

In [179]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "svc__C": [0.1, 1, 10, 100, 1000],
    "svc__kernel": ["rbf", "linear"],
    "svc__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
y_pred_tuned = grid.predict(X_test)
y_pred_proba_tuned = grid.predict_proba(X_test)

In [180]:
print(grid.best_params_)
print(grid.best_score_)

{'svc__C': 10, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
0.8094791535475098


In [189]:
from sklearn.metrics import classification_report, roc_auc_score

print('ROC AUC score: ', roc_auc_score(y_test, y_pred_proba_tuned, multi_class='ovr', average='weighted'))
print('\nClassification report:')
print(classification_report(y_test, y_pred_tuned))

ROC AUC score:  0.8940360085505853

Classification report:
              precision    recall  f1-score   support

           0       0.69      0.56      0.62        39
           1       0.87      0.93      0.90       175
           2       0.76      0.62      0.68        26

    accuracy                           0.84       240
   macro avg       0.77      0.70      0.73       240
weighted avg       0.83      0.84      0.83       240



Cross Validation

In [182]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
roc_auc_scores = []
classification_reports = []

for fold, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold+1}/{skf.get_n_splits()}")

    X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    model.fit(X_train_fold, y_train_fold)

    y_pred_proba_fold = model.predict_proba(X_val_fold)
    y_pred_fold = model.predict(X_val_fold)

    roc_auc = roc_auc_score(y_val_fold, y_pred_proba_fold, multi_class='ovr', average='weighted')
    roc_auc_scores.append(roc_auc)
    print(f"ROC AUC Score: {roc_auc:.4f}")

    class_report = classification_report(y_val_fold, y_pred_fold, output_dict=True)
    classification_reports.append(class_report)
    print("Classification Report:\n", classification_report(y_val_fold, y_pred_fold))


Fold 1/10
ROC AUC Score: 0.8777
Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.88      0.72        16
           1       0.92      0.81      0.86        70
           2       0.55      0.60      0.57        10

    accuracy                           0.80        96
   macro avg       0.69      0.76      0.72        96
weighted avg       0.83      0.80      0.81        96


Fold 2/10
ROC AUC Score: 0.8778
Classification Report:
               precision    recall  f1-score   support

           0       0.48      0.75      0.59        16
           1       0.90      0.77      0.83        70
           2       0.55      0.60      0.57        10

    accuracy                           0.75        96
   macro avg       0.64      0.71      0.66        96
weighted avg       0.79      0.75      0.76        96


Fold 3/10
ROC AUC Score: 0.8468
Classification Report:
               precision    recall  f1-score   support

           0

In [185]:
cv_results = cross_validate(model, X_train, y_train, cv=skf,
                            scoring=['f1_weighted', 'roc_auc_ovr_weighted'],
                            n_jobs=-1)

print(f"Mean F1 Weighted: {cv_results['test_f1_weighted'].mean():.4f}")
print(f"Mean ROC AUC: {cv_results['test_roc_auc_ovr_weighted'].mean():.4f}")

Mean F1 Weighted: 0.7777
Mean ROC AUC: 0.8787
